# 00 — Étiquetage supervisé des jardins (FR / EN)
**Objectif :** attribuer manuellement une catégorie indépendante des neuf observations Gemini aux 31 photos. Cette catégorie est la vérité terrain de KNN.

Classes proposées : `LAWN_DOMINANT` (pelouse dominante), `TERRACE_DOMINANT` (terrasse / surfaces aménagées dominantes), `MIXED` (jardin mixte).

**Important :** les neuf indicateurs ne permettent pas de calculer la surface dominante. Examinez chaque photo avant de choisir. Aucune catégorie n'est préremplie arbitrairement. Les insertions ne se produisent qu'après un clic explicite.


In [ ]:
from pathlib import Path
import os, json, psycopg, pandas as pd
from dotenv import load_dotenv
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai-service/plant-ingestion").exists())
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT / ".env.local", override=True)
dsn = os.environ["GARDNAIZE_ML_DATABASE_URL"]
dsn = dsn.replace("postgresql+psycopg://","postgresql://",1)
FEATURES = ("lawn","terrace","pathway","trees","shrubs","fence","low_wall","stairs","slope")
def conn(): return psycopg.connect(dsn)
from IPython.display import display, Image, Markdown
import ipywidgets as widgets
with conn() as c:
    with c.cursor() as cur:
        cur.execute("""SELECT g.image_id,g.image_uri,f.feature_values
        FROM horticulture.garden_image_candidate g
        JOIN horticulture.ml_garden_features f ON f.image_id=g.image_id
        WHERE g.review_status='ACCEPTED' AND f.reviewed=true ORDER BY g.image_id""")
        gardens=cur.fetchall()
        cur.execute("SELECT image_id,label_value FROM horticulture.ml_garden_label WHERE label_type='GARDEN_TYPE'")
        existing=dict(cur.fetchall())
print("Jardins validés :",len(gardens),"| Labels existants :",len(existing))



In [ ]:
choices=[("— Choisir / Select —",""),("Pelouse dominante / Lawn dominant","LAWN_DOMINANT"),("Terrasse dominante / Terrace dominant","TERRACE_DOMINANT"),("Mixte / Mixed","MIXED")]
selectors={}
for image_id,uri,features in gardens:
    p=(ROOT/"ai-service/plant-ingestion"/uri).resolve()
    allowed=(ROOT/"ai-service/plant-ingestion/inputs/images").resolve()
    display(Markdown(f"### Jardin / Garden #{image_id}"))
    if p.is_relative_to(allowed) and p.is_file():
        display(Image(filename=str(p),width=500))
    else:
        print("Image absente ou hors dossier autorisé :",uri)
    print("Observations Gemini :",", ".join(f"{k}={features.get(k)}" for k in FEATURES))
    w=widgets.Dropdown(options=choices,value=existing.get(image_id,""),description="Classe :",
                       style={"description_width":"initial"},layout=widgets.Layout(width="500px"))
    selectors[image_id]=w
    display(w)



In [ ]:
save=widgets.Button(description="Enregistrer les labels / Save labels",button_style="success")
status=widgets.Output()
def save_labels(_):
    with status:
        status.clear_output()
        missing=[i for i,w in selectors.items() if not w.value]
        if missing:
            print("Aucun enregistrement : catégories manquantes :",missing)
            return
        rows=[(i,"GARDEN_TYPE",w.value,"HUMAN") for i,w in selectors.items()]
        with conn() as c:
            with c.cursor() as cur:
                from psycopg import sql
                placeholders=sql.SQL(",").join([sql.SQL("(%s,%s,%s,%s)")] * len(rows))
                statement=sql.SQL("""INSERT INTO horticulture.ml_garden_label
                    (image_id,label_type,label_value,label_source) VALUES {}
                    ON CONFLICT (image_id,label_type) DO UPDATE SET
                    label_value=EXCLUDED.label_value,label_source=EXCLUDED.label_source,reviewed_at=now()""").format(placeholders)
                cur.execute(statement,[value for row in rows for value in row])
            c.commit()
        print(f"{len(rows)} labels enregistrés dans une transaction (une seule requête SQL paramétrée).")
save.on_click(save_labels)
display(save,status)




In [ ]:
with conn() as c:
    df=pd.read_sql_query("""SELECT label_value,COUNT(*) AS n FROM horticulture.ml_garden_label
    WHERE label_type='GARDEN_TYPE' GROUP BY label_value ORDER BY label_value""",c)
display(df)

